# Equivariant Filter Basics

This notebook introduces the Equivariant Filter (EqF) framework through simple examples and visualizations.

## Learning Objectives

- Understand the concept of equivariance in filtering
- Learn how EqF generalizes IEKF
- Implement a simple equivariant observer
- Visualize the benefits of equivariant design

## Prerequisites

- Familiarity with IEKF (see notebook 01)
- Understanding of group actions
- Python and NumPy knowledge

In [ ]:
# Install required packages
# !pip install numpy scipy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. What is Equivariance?

A function f is **equivariant** with respect to a group action if:

```
f(g · x) = g · f(x)
```

For example, if we scale all inputs, the output scales accordingly.

### Example: Rotation Equivariance

In [ ]:
def rotation_matrix_2d(theta):
    """2D rotation matrix."""
    return np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta), np.cos(theta)]
    ])

# A simple equivariant function: norm
x = np.array([1.0, 0.0])
theta = np.pi / 4
R = rotation_matrix_2d(theta)

print(f"Original vector: {x}")
print(f"Norm: {np.linalg.norm(x)}")
print(f"\nRotated vector: {R @ x}")
print(f"Norm after rotation: {np.linalg.norm(R @ x)}")
print(f"\nNorm is rotation-invariant!")

## 2. Equivariant Systems

An equivariant system satisfies:

**Dynamics:** ẋ = f(x, u) where f(g·x, u) = g·f(x, u)

**Output:** y = h(x) where h(g·x) = ρ(g)·h(x)

### Example: Particle on a Sphere

In [ ]:
# State: point on unit sphere S²
# Dynamics: tangent to sphere (velocity)
# Group: SO(3) rotations

def normalize(v):
    """Normalize vector to unit length."""
    return v / np.linalg.norm(v)

def sphere_dynamics(x, v, dt):
    """Move on sphere with tangent velocity."""
    # Project velocity to tangent space
    v_tangent = v - np.dot(v, x) * x
    # Move and project back to sphere
    x_new = x + v_tangent * dt
    return normalize(x_new)

# Simulate motion on sphere
T = 5.0
dt = 0.01
N = int(T / dt)

# Initial state
x = normalize(np.array([1.0, 0.0, 0.0]))
v = np.array([0.0, 1.0, 0.5])  # tangent velocity

trajectory = np.zeros((N, 3))
for i in range(N):
    trajectory[i] = x
    x = sphere_dynamics(x, v, dt)

print(f"Simulated {N} steps on sphere")
print(f"All points on unit sphere: {np.allclose(np.linalg.norm(trajectory, axis=1), 1.0)}")

In [ ]:
# Visualize trajectory on sphere
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Draw sphere
u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0, np.pi, 50)
x_sphere = np.outer(np.cos(u), np.sin(v))
y_sphere = np.outer(np.sin(u), np.sin(v))
z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))
ax.plot_surface(x_sphere, y_sphere, z_sphere, alpha=0.1, color='gray')

# Plot trajectory
ax.plot(trajectory[:, 0], trajectory[:, 1], trajectory[:, 2], 
        'b-', linewidth=2, label='Trajectory')
ax.scatter([trajectory[0, 0]], [trajectory[0, 1]], [trajectory[0, 2]], 
           c='g', s=100, label='Start')
ax.scatter([trajectory[-1, 0]], [trajectory[-1, 1]], [trajectory[-1, 2]], 
           c='r', s=100, label='End')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Motion on Unit Sphere')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Equivariant Observer Design

The EqF constructs an observer that respects system equivariance:

1. **Lift** the problem to a Lie group G
2. Define **equivariant error** on G
3. Design **equivariant innovation**
4. **Update** preserving equivariance

### Simple 2D Example

In [ ]:
# State: position in R² (with SE(2) symmetry)
# Measurement: range from origin

class SimpleEqF:
    def __init__(self, x0, P0, Q, R):
        self.x = x0.copy()  # Position estimate
        self.P = P0.copy()  # Covariance
        self.Q = Q          # Process noise
        self.R = R          # Measurement noise
    
    def predict(self, v, dt):
        """Predict with velocity input."""
        self.x = self.x + v * dt
        self.P = self.P + self.Q * dt
    
    def update(self, measurement):
        """Update with range measurement."""
        # Predicted measurement
        range_pred = np.linalg.norm(self.x)
        
        # Innovation
        innovation = measurement - range_pred
        
        # Measurement Jacobian (direction to origin)
        if range_pred > 1e-6:
            H = self.x / range_pred
        else:
            H = np.zeros(2)
        
        # Kalman gain
        S = H @ self.P @ H + self.R
        K = self.P @ H / S
        
        # Update
        self.x = self.x + K * innovation
        self.P = (np.eye(2) - np.outer(K, H)) @ self.P

print("Simple EqF class defined")

## 4. Simulation and Comparison

In [ ]:
# Simulation parameters
T = 10.0
dt = 0.05
N = int(T / dt)

# True trajectory (circular motion)
omega = 0.5  # angular velocity
radius = 2.0

x_true = np.zeros((N, 2))
measurements = np.zeros(N)

for i in range(N):
    t = i * dt
    x_true[i] = radius * np.array([np.cos(omega * t), np.sin(omega * t)])
    # Noisy range measurement
    measurements[i] = np.linalg.norm(x_true[i]) + np.random.normal(0, 0.1)

# Initialize filter with error
x0 = np.array([3.0, 0.5])
P0 = np.eye(2)
Q = 0.01 * np.eye(2)
R_meas = 0.01

eqf = SimpleEqF(x0, P0, Q, R_meas)

# Run filter
x_est = np.zeros((N, 2))
errors = np.zeros(N)

for i in range(N):
    if i > 0:
        # Compute velocity from true trajectory
        v = (x_true[i] - x_true[i-1]) / dt
        eqf.predict(v, dt)
    
    # Update
    eqf.update(measurements[i])
    
    # Store
    x_est[i] = eqf.x
    errors[i] = np.linalg.norm(x_true[i] - x_est[i])

print(f"Mean error: {np.mean(errors):.4f}")
print(f"Final error: {errors[-1]:.4f}")

In [ ]:
# Visualize results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Trajectory plot
ax1.plot(x_true[:, 0], x_true[:, 1], 'g-', linewidth=2, label='True')
ax1.plot(x_est[:, 0], x_est[:, 1], 'b--', linewidth=2, label='Estimate')
ax1.scatter([x_true[0, 0]], [x_true[0, 1]], c='g', s=100, marker='o', label='Start')
ax1.scatter([0], [0], c='k', s=100, marker='x', label='Origin')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_title('Trajectories')
ax1.legend()
ax1.grid(True)
ax1.axis('equal')

# Error plot
time = np.arange(N) * dt
ax2.plot(time, errors, 'r-', linewidth=2)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Position Error')
ax2.set_title('Estimation Error')
ax2.grid(True)

plt.tight_layout()
plt.show()

## 5. Key Insights

### Equivariance Benefits

1. **Geometric Consistency**: The filter respects the natural geometry
2. **Reduced Bias**: Linearization errors are distributed symmetrically
3. **Better Convergence**: Faster convergence from initialization errors

### When to Use EqF

✅ **Good for:**
- Systems with clear symmetries
- State spaces that are homogeneous spaces
- Visual-inertial odometry
- SLAM problems

❌ **Might be overkill for:**
- Simple linear systems
- Systems without obvious symmetries
- When computational simplicity is paramount

## 6. Comparison: EqF vs Standard EKF

| Property | Standard EKF | Equivariant Filter |
|----------|--------------|--------------------|
| Error definition | Additive (x - x̂) | Equivariant (g·x, ĝ·x) |
| Linearization | Local | Respects symmetry |
| Consistency | Can drift | Better maintained |
| Complexity | Lower | Higher |
| Flexibility | Limited | High |

## 7. Advanced Topics

### Homogeneous Spaces
- Quotient spaces G/H
- Natural for many robotics problems
- EqF provides systematic design method

### Lifted Representations
- Represent state on larger Lie group
- Design filter in lifted space
- Project back to original space

### Output Equivariance
- Measurements may have their own symmetries
- EqF handles this naturally
- Key for visual measurements

## Next Steps

1. **Extend to SE(3)**: Full 3D pose estimation
2. **Add landmarks**: SLAM applications
3. **Visual measurements**: Camera observations
4. **Compare with IEKF**: Understand the differences

## References

- van Goor et al. (2021). "Equivariant Filter (EqF)". [arXiv:2010.14666](https://arxiv.org/abs/2010.14666)
- [GTSAM EKF Variants](https://borglab.github.io/gtsam/ekf-variants/)
- [AwesomeEqF Resources](https://github.com/borglab/AwesomeEqF)